# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://01517885-5c59-4d08-8ed4-17006164d017.us-east-1-1.aws.cloud.qdrant.io


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [3]:
from langchain_core.documents import Document
import fitz

# 처리할 PDF 파일 목록
pdf_files = [
    "../datasets/IoT_공통보안가이드.pdf",
    "../datasets/KISA_홈가전IoT보안가이드.pdf",
    "../datasets/사물인터넷(IoT)_환경에서의_암호_인증기술_이용_안내서(17년_개정본).pdf"
]

docs = []

# 각 PDF를 페이지 단위의 Document로 변환 (Parent Document)
for file_path in pdf_files:
    doc = fitz.open(file_path)
    source_name = file_path.split("/")[-1]

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text", sort=True)

        # 빈 페이지는 스킵
        if len(text.strip()) < 10:
            continue

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": source_name,
                    "page": page_num + 1,
                    "parent_id": f"{source_name}_page_{page_num + 1}"
                }
            )
        )

    doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 363개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 65자
평균 페이지 길이: 1849자

첫 페이지 내용 미리보기:
   IoT Common Security Guide

ICT 융합 제품·서비스의
보안 내재화를 위한
공통 보안 가이드...


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 개선 1: 청킹 전략 최적화 (400/50 -> 500/100, 문맥 보존 강화)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# 개선 4: 메타데이터 활용 (키워드 기반 카테고리 태깅)
CATEGORY_KEYWORDS = {
    "하드웨어_물리보안": ["분해", "디버그", "물리적", "외부 포트", "탬퍼", "회로 접근"],
    "인증_접근통제": ["인증", "비밀번호", "접근권한", "비인가"],
    "암호화_데이터보호": ["암호화", "암호키", "개인정보", "중요정보", "데이터 보호"],
    "펌웨어_플랫폼보안": ["펌웨어", "업데이트", "부팅", "패치", "취약점"],
    "인터페이스_통신설계": ["UART", "JTAG", "SPI", "I2C", "USB", "GPIO", "통신"],
    "회로_전원_신호설계": ["전원", "리셋", "클록", "신호", "저전력"],
    "PCB_배선_기판설계": ["PCB", "배선", "테스트 포인트", "기판"],
    "MCU_메모리_부품설계": ["MCU", "메모리", "저장장치", "센서"],
}

def assign_category(text: str) -> str:
    """키워드 매칭으로 chunk에 카테고리를 태깅 (LLM 호출 없이 빠르게 처리)"""
    scores = {cat: sum(1 for kw in kws if kw.lower() in text.lower()) for cat, kws in CATEGORY_KEYWORDS.items()}
    best_cat = max(scores, key=scores.get)
    return best_cat if scores[best_cat] > 0 else "기타"

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"],
                    "category": assign_category(chunk)
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# 카테고리 분포 확인
from collections import Counter
category_counts = Counter(d.metadata["category"] for d in child_docs)
print(f"\n카테고리 분포:")
for cat, cnt in category_counts.most_common():
    print(f"  - {cat}: {cnt}개")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Category: {child_docs[i].metadata['category']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 363
  - Child chunk 수: 2051
  - 평균 chunk/page: 5.7

카테고리 분포:
  - 기타: 1112개
  - 인증_접근통제: 309개
  - 암호화_데이터보호: 227개
  - 펌웨어_플랫폼보안: 179개
  - 인터페이스_통신설계: 87개
  - 하드웨어_물리보안: 71개
  - MCU_메모리_부품설계: 56개
  - 회로_전원_신호설계: 10개

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: IoT_공통보안가이드.pdf_page_1
  Page: 1
  Category: 기타
  Length: 62자
  Content: IoT Common Security Guide

ICT 융합 제품·서비스의
보안 내재화를 위한
공통 보안 가이드...

Chunk 2:
  Parent ID: IoT_공통보안가이드.pdf_page_3
  Page: 3
  Category: 기타
  Length: 73자
  Content: 1      2016. 9           ICT 융합 제품·서비스의 보안 내재화를 위한 IoT 공통 보안 가이드

2

3

4...

Chunk 3:
  Parent ID: IoT_공통보안가이드.pdf_page_4
  Page: 4
  Category: 인증_접근통제
  Length: 453자
  Content: Contents


        I              1. 배경 및 범위 / 8
  개요            2. IoT 보안위협 / 9

                  ...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://01517885-5c59-4d08-8ed4-17006164d017.us-east-1-1.aws.cloud.qdrant.io


In [6]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "IoT 디바이스 보안 점검 도우미"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'IoT 디바이스 보안 점검 도우미'이 이미 존재합니다.
컬렉션 'IoT 디바이스 보안 점검 도우미' 삭제 중...
컬렉션이 삭제되었습니다.
컬렉션 'IoT 디바이스 보안 점검 도우미' 생성 완료

2051개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [7]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 363개의 Parent 문서 저장 완료

Docstore 키 예시: ['IoT_공통보안가이드.pdf_page_1', 'IoT_공통보안가이드.pdf_page_3', 'IoT_공통보안가이드.pdf_page_4', 'IoT_공통보안가이드.pdf_page_5', 'IoT_공통보안가이드.pdf_page_6']


## 5. Parent Document Retriever 구현

In [8]:
from typing import List
import re

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 3):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k  # 개선 2: 검색 개수(k) 2 -> 3으로 확대, 회수율 향상

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

    def hybrid_search(self, query: str, k: int = 3, fetch_k: int = 15) -> List[Document]:
        """
        개선 5: 하이브리드 검색 (벡터 검색 + 키워드 매칭 재정렬)
        1. 벡터 검색으로 fetch_k개 후보를 넉넉히 가져온 뒤
        2. 질문의 키워드가 chunk에 얼마나 등장하는지로 재점수화하여 상위 k개 선택
        """
        candidates = self.vectorstore.similarity_search(query, k=fetch_k)
        keywords = [w for w in re.findall(r"[가-힣A-Za-z0-9]+", query) if len(w) > 1]

        def keyword_score(text: str) -> int:
            return sum(text.count(kw) for kw in keywords)

        # 벡터 검색 순위(먼저 나온 순서=유사도 높음)와 키워드 점수를 함께 반영
        scored = [
            (idx, keyword_score(doc.page_content), doc)
            for idx, doc in enumerate(candidates)
        ]
        # 키워드 점수 내림차순 -> 동점이면 원래 벡터 순위(idx) 오름차순
        scored.sort(key=lambda x: (-x[1], x[0]))

        return [doc for _, _, doc in scored[:k]]

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=3
)

print("✓ Parent Document Retriever 생성 완료 (k=3, 하이브리드 검색 지원)")

✓ Parent Document Retriever 생성 완료 (k=3, 하이브리드 검색 지원)


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [9]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "집에 설치한 웹캠이 혼자 움직이는데, 해킹당한 건지 어떻게 확인하고 대처해야 하나요?"

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 집에 설치한 웹캠이 혼자 움직이는데, 해킹당한 건지 어떻게 확인하고 대처해야 하나요?


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 132
  Parent ID: KISA_홈가전IoT보안가이드.pdf_page_132
  길이: 427자
  내용: 네트워크 카메라                  http://www.uplusiotshop.com/MMall/
                                                                                                                                                                                                                                                    보안위협
스마트라우터                     https://www.getcujo.com/, www.lge.com

가정용 방화벽                    https://www.getcujo.com/

Chunk 2:
  페이지: 18
  Parent ID: KISA_홈가전IoT보안가이드.pdf_page_18
  길이: 346자
  내용: (정의) 댁내에 설치된 네트워크 카메라를 통하여 영상을 촬영하여 저장하거나 촬영한       영상정보를 네트워크 통신채널로 전송하는 제품                                                                                                                                         대응방안
                                  

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from typing import Optional
from qdrant_client import models

llm = init_chat_model("gpt-5.4-mini")

# 카테고리 분류 결과를 위한 Pydantic 모델
class CategoryClassification(BaseModel):
    """IoT 보안 점검 카테고리 분류 결과"""
    category: Optional[str] = Field(
        description="선택된 카테고리 이름. 적합한 카테고리가 없으면 None"
    )

def determine_category(question: str) -> Optional[str]:
    """
    LLM을 사용하여 질문을 분석하고 적절한 IoT 보안 점검 카테고리를 결정합니다.

    Args:
        question: 사용자 질문

    Returns:
        카테고리 이름 (문자열) 또는 None (필터 없음)
    """

    # 사용 가능한 카테고리 목록
    available_categories = {
    "하드웨어_물리보안": "제품 분해, 디버그 포트 노출, 내부 회로 접근, 물리적 조작 및 하드웨어 보호, 외부 포트 차단 관련",
    "인증_접근통제": "사용자 인증, 기기 간 인증, 비밀번호 관리, 접근권한, 비인가 사용자나 장치의 접근 차단 관련",
    "암호화_데이터보호": "개인정보와 중요정보 보호, 저장·전송 데이터 암호화, 암호키 관리, 데이터 무결성, 안전한 통신 관련",
    "펌웨어_플랫폼보안": "펌웨어 업데이트, 소프트웨어 취약점, 안전한 부팅, 보안패치, 플랫폼 보호 관련",
    "인터페이스_통신설계": "UART, JTAG, SPI, I2C, USB, GPIO 등 인터페이스, 통신 구조 및 네트워크 보안 관련",
    "회로_전원_신호설계": "전원, 리셋, 클록, 신호 안정성, 저전력 설계 등 기기 동작 안정성 관련",
    "PCB_배선_기판설계": "PCB 구조, 부품 배치, 배선, 테스트 포인트, 기판 설계 관련",
    "MCU_메모리_부품설계": "MCU, 메모리, 저장장치, 센서, 하드웨어 모듈 구성 관련",
}

    # LLM에게 카테고리 분류 요청
    category_list = "\n".join([f"- {cat}: {desc}" for cat, desc in available_categories.items()])

    classification_prompt = f"""다음 질문을 분석하여 가장 적합한 IoT 보안 점검 카테고리를 선택하세요.

<available_categories>
{category_list}
</available_categories>

<question>
{question}
</question>

<rules>
1. 질문의 주요 주제와 가장 관련 있는 카테고리를 선택하세요
2. 여러 카테고리가 관련될 수 있지만, 가장 핵심적인 하나만 선택하세요
3. 적합한 카테고리가 없거나 매우 일반적인 질문이면 category를 null로 설정하세요
</rules>
"""

    # Structured Output을 사용하여 LLM 호출
    structured_llm = llm.with_structured_output(CategoryClassification)
    result = structured_llm.invoke(classification_prompt)

    print(f"[LLM 분류 결과]")
    print(f"  카테고리: {result.category}")

    return result.category


def rag_with_dynamic_filter(question: str) -> str:
    """
    개선 4: 메타데이터(category) 기반 동적 필터링을 적용한 RAG
    """
    # 1. 질문 분석하여 카테고리 결정
    category = determine_category(question)

    # 2. 필터 설정
    search_kwargs = {"k": 3}
    if category:
        search_kwargs["filter"] = models.Filter(
            must=[
                models.FieldCondition(
                    key="metadata.category",
                    match=models.MatchValue(value=category)
                )
            ]
        )
        print(f"✓ 적용된 필터: category = '{category}'\n")
    else:
        print(f"✓ 필터 없음 (전체 문서 검색)\n")

    # 3. 문서 검색
    retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
    retrieved_docs = retriever.invoke(question)

    # 4. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        page = doc.metadata.get('page', '?')
        cat = doc.metadata.get('category', '?')
        context_parts.append(
            f"[출처: {doc.metadata.get('source', '?')}, 페이지: {page}, 카테고리: {cat}]\n{doc.page_content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # 5. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)
    return response.content


def rag_with_hybrid_search(question: str) -> str:
    """
    개선 5: 하이브리드 검색(벡터+키워드)을 적용한 RAG
    """
    # 1. 하이브리드 검색으로 child chunk 확보
    retrieved_docs = parent_retriever.hybrid_search(question, k=3)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성 및 LLM 호출
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)
    return response.content


template = """
당신은 'IoT 디바이스 보안 점검 도우미'입니다.

집에서 사용하는 웹캠, 스마트 도어락, 공유기 같은 스마트 기기의
보안을 점검하고, 이상 증상에 대처하는 방법을 안내합니다.

사용자는 IT 전문가가 아닌 일반인입니다.
답변은 최대한 쉽고 친근한 한국어로 작성하세요.


[중요 원칙]

1. 반드시 아래 [참고 정보]에 있는 내용만 근거로 답하세요.
   참고 정보에 없는 내용을 만들어내지 마세요.
   자료만으로 판단이 어려우면
   "현재 참고 자료만으로는 이 부분을 정확하게 안내하기 어렵습니다."
   라고 안내하세요.

2. 전문용어는 가능한 한 사용하지 마세요.
   꼭 필요한 경우에만 괄호 안에 넣으세요.

   예:
   - "기기 소프트웨어(펌웨어)를 최신 버전으로 업데이트하세요."
   - "무선 암호화 방식(WPA2 이상)을 사용하세요."


[이상 증상 질문 답변 방법]

사용자가 기기의 이상한 동작을 물어보면:

1. 공감 및 상황 설명
   사용자의 걱정에 공감하고,
   해당 증상이 왜 발생할 수 있는지 쉽게 설명

2. 즉시 할 수 있는 조치
   비전문가도 바로 실행할 수 있는 대처 방법 안내
   (전원 끄기, 비밀번호 변경, 초기화, 네트워크 분리 등)

3. 보안 점검 방법
   참고 자료에 관련 보안 가이드가 있다면
   해당 기기의 보안 점검 포인트를 쉽게 안내

4. 추가 도움
   필요한 경우 신고 기관이나 전문가 도움을 받을 수 있는 곳 안내


예:

나쁜 답변:
"JTAG 포트를 비활성화하고 RDP Level 2를 적용하세요."

좋은 답변:
"웹캠이 혼자 움직인다면 외부에서 누군가 접근했을 가능성이 있습니다.
우선 웹캠의 전원을 뽑고, 공유기 비밀번호를 변경하세요.
그 다음 웹캠 앱에서 연결된 기기 목록을 확인해서
모르는 기기가 있으면 삭제하세요."


[보안 점검 질문 답변 방법]

사용자가 보안 점검 방법을 물어보면:

1. 기본 보안 수칙을 체크리스트 형태로 안내
2. 해당 기기에 맞는 구체적인 설정 방법 설명
3. 참고 자료의 보안 가이드 내용을 쉽게 풀어서 설명

기본 보안 수칙 예:
- 초기 비밀번호를 반드시 변경했는지
- 기기 소프트웨어(펌웨어)를 최신 버전으로 업데이트했는지
- 사용하지 않는 기능(원격 접속 등)을 껐는지
- 공유기 암호화 방식이 안전한지 (WPA2/WPA3)


[신고/도움 요청 질문 답변 방법]

사용자가 신고 방법을 물어보면:

1. 상황에 맞는 신고 기관 안내
2. 연락처와 신고 방법을 구체적으로 안내
3. 신고 전에 준비할 것(증거 보존 등) 안내


[제품별 안내]

사용자가 특정 스마트 기기를 말하면
그 제품의 특성을 고려하여 안내하세요.

예:
- 웹캠/홈캠 → 영상 유출, 원격 접속, 카메라 제어 위험
- 스마트 도어락 → 출입 통제, 비밀번호 관리, 물리적 접근
- 공유기 → 네트워크 전체 보안, 접속 기기 관리
- 스마트TV → 계정 보안, 카메라·마이크 보호
- 스마트 플러그/센서 → 원격 제어, 데이터 변조


[답변 형식]

### 상황 판단
사용자의 상황이 왜 발생할 수 있는지 1~2문장으로 설명

### 지금 바로 할 수 있는 조치
비전문가도 실행 가능한 구체적인 행동 지침을 순서대로 안내

### 보안 점검 체크리스트
해당 기기에 맞는 보안 점검 항목 안내

### 추가 도움이 필요하면
신고 기관, 전문가 상담 등 추가 지원 안내

### 참고 자료
실제로 답변에 사용한 문서명과 페이지 번호만 표시

예:
- IoT 공통보안가이드, p.20
- 홈·가전 IoT 보안가이드, p.42


[참고 정보]
{context}


[사용자 질문]
{question}


[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)


def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content


print("✓ RAG 시스템 준비 완료 (기본 / 동적 필터 / 하이브리드 검색 지원)")

✓ RAG 시스템 준비 완료 (기본 / 동적 필터 / 하이브리드 검색 지원)


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [11]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "집에 설치한 웹캠이 혼자 움직이는데 해킹당한 건가요?",

    "공유기 비밀번호를 한 번도 안 바꿨는데 위험한가요?",

    "스마트 도어락이 해킹당하면 어떻게 대처해야 하나요?",

    "집에 있는 스마트 기기 보안 점검은 어떻게 하나요?",

    "IoT 기기가 해킹당한 것 같으면 어디에 신고하나요?",

    "공유기 보안 설정을 안전하게 바꾸려면 어떻게 해야 하나요?"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 집에 설치한 웹캠이 혼자 움직이는데 해킹당한 건가요?



### 상황 판단
웹캠이 혼자 움직인다면, 누군가가 원격으로 제어하고 있거나 설정이 바뀌었을 가능성을 걱정해볼 수 있습니다. 홈캠(웹캠)은 영상이 외부로 노출되거나, 인증정보가 유출되면 보안에 취약해질 수 있습니다.

### 지금 바로 할 수 있는 조치
1. 웹캠의 전원을 먼저 꺼 주세요.
2. 웹캠이 연결된 앱이나 계정에서 모르는 기기가 있는지 확인해 주세요.
3. 웹캠에 연결된 비밀번호가 있다면 바꿔 주세요.
4. 가능하면 웹캠을 인터넷 연결에서 잠시 분리해 두세요.
5. 웹캠 앱에서 최근 저장된 영상이나 접속 기록이 보이면 확인해 주세요.

### 보안 점검 체크리스트
- 초기 비밀번호를 그대로 쓰고 있지 않은지 확인
- 웹캠 계정 비밀번호를 바꿨는지 확인
- 웹캠 소프트웨어(펌웨어)를 최신 상태로 유지하고 있는지 확인
- 인가된 사용자만 영상에 접근할 수 있게 되어 있는지 확인
- 모르는 기기나 계정이 연결돼 있지 않은지 확인

웹캠은 참고 자료상 “촬영 제품”에 해당하며, 개인영상 유출이나 위장이 보안위협으로 제시되어 있습니다. 그래서 이상 동작이 보이면 영상 접근 권한과 인증정보를 먼저 점검하는 것이 중요합니다.

### 추가 도움이 필요하면
현재 참고 자료만으로는 이 상황에서 바로 신고해야 하는 기관이나 연락처를 정확하게 안내하기 어렵습니다.  
다만 전원이 꺼진 뒤에도 이상이 계속되거나, 저장된 영상이 밖으로 나간 흔적이 보인다면 제조사 고객센터나 보안 점검이 가능한 전문가의 도움을 받는 것이 좋습니다.

### 참고 자료
- KISA_홈가전IoT보안가이드.pdf, p.13
- KISA_홈가전IoT보안가이드.pdf, p.18


질문: 공유기 비밀번호를 한 번도 안 바꿨는데 위험한가요?



### 상황 판단
네, 위험할 수 있습니다. 참고 자료에 따르면 초기 비밀번호를 그대로 쓰거나, 예측하기 쉬운 비밀번호를 쓰면 비인가된 사용자가 접근할 가능성이 커집니다.

### 지금 바로 할 수 있는 조치
1. 공유기 비밀번호를 바로 바꾸세요.  
2. 새 비밀번호는 너무 짧지 않게, **영문자·숫자·특수문자를 섞어서 8자리 이상**으로 설정하세요.  
3. `admin`, `root`, `pass`처럼 **쉽게 떠올릴 수 있는 비밀번호는 피하세요.**  
4. 이전에 쓰던 비밀번호와 **같은 비밀번호는 다시 쓰지 마세요.**

### 보안 점검 체크리스트
- 초기 비밀번호를 그대로 사용하고 있지 않은가요?
- 비밀번호가 너무 짧거나 단순하지 않은가요?
- `admin`, `root`, `pass` 같은 쉬운 비밀번호를 쓰고 있지 않은가요?
- 이전 비밀번호를 다시 사용하고 있지 않은가요?
- 공유기 관리 화면 접근에 사용자 인증이 필요한가요?

### 추가 도움이 필요하면
현재 참고 자료만으로는 공유기의 세부 설정 방법이나 신고 기관까지 정확하게 안내하기 어렵습니다.  
다만 비밀번호 변경 후에도 모르는 접속 흔적이 계속 보인다면, 제조사 고객센터나 인터넷 서비스 제공업체에 도움을 요청하는 것이 좋습니다.

### 참고 자료
- KISA_홈가전IoT보안가이드.pdf, p.58
- KISA_홈가전IoT보안가이드.pdf, p.60


질문: 스마트 도어락이 해킹당하면 어떻게 대처해야 하나요?



...

### 상황 판단
집에 있는 스마트 기기는 초기 설정 그대로 쓰거나, 업데이트를 안 하면 보안에 취약해질 수 있습니다.  
현재 참고 자료에는 제목 정보만 있고, 실제 점검 항목이나 설정 방법이 보이지 않아 구체적으로 안내하기 어렵습니다.

### 지금 바로 할 수 있는 조치
현재 참고 자료만으로는 이 부분을 정확하게 안내하기 어렵습니다.  
다만, 스마트 기기 설명서나 앱에서 아래 같은 기본 항목을 먼저 확인해 보세요.
- 초기 비밀번호를 바꿀 수 있는지
- 기기 소프트웨어(펌웨어) 업데이트가 있는지
- 원격 접속 같은 기능을 끌 수 있는지
- 모르는 기기가 연결되어 있지 않은지

### 보안 점검 체크리스트
현재 참고 자료만으로는 기기별 보안 점검 항목을 정확하게 안내하기 어렵습니다.  
참고 자료에 실제 내용이 확인되면, 그에 맞춰 쉽게 정리해드릴 수 있습니다.

### 추가 도움이 필요하면
스마트 기기에서 이상한 동작이 계속되거나, 계정이 털린 것 같다면 제조사 고객센터에 먼저 문의하는 것이 좋습니다.  
또한 현재 참고 자료만으로는 신고 기관 안내까지 정확히 연결하기 어렵습니다.

### 참고 자료
- KISA_홈가전IoT보안가이드.pdf, p.5
- KISA_홈가전IoT보안가이드.pdf, p.11
- KISA_홈가전IoT보안가이드.pdf, p.15


질문: IoT 기기가 해킹당한 것 같으면 어디에 신고하나요?



### 상황 판단
IoT 기기가 해킹당한 것 같다면, 먼저 해당 제품/서비스의 보안취약점 정보와 해결 방법을 확인하는 것이 중요합니다. 참고 자료에는 이런 정보가 제품 홈페이지나 등록한 고객에게 보내는 이메일 같은 방식으로 신속히 제공되어야 한다고 되어 있습니다.

### 지금 바로 할 수 있는 조치
1. 사용 중인 기기 제조사 또는 서비스 제공 회사의 홈페이지를 확인하세요.  
2. 제품을 등록해 둔 적이 있다면, 안내 메일이 왔는지 확인하세요.  
3. 보안취약점 안내에 있는 패치, 업데이트, 우회 방법이 있으면 바로 따라 하세요.  
4. 보안 대응 담당자 연락처가 공개되어 있다면 그쪽으로 문의하세요.  
5. 가능하면 기기 모델명, 모델번호, OS 종류/버전, 프로토콜 버전 같은 정보를 함께 확인해 두세요.

### 보안 점검 체크리스트
- 기기/서비스의 보안취약점 정보가 홈페이지에 공개되어 있는지 확인
- 등록한 사용자에게 이메일로 안내가 왔는지 확인
- 패치나 업데이트 같은 해결책이 있는지 확인
- 우회 방법이 안내되어 있는지 확인
- 보안취약점 대응 담당자 연락처가 공개되어 있는지 확인
- 회사의 보안취약점 대응 정책이 공개되어 있는지 확인

### 추가 도움이 필요하면
현재 참고 자료만으로는, 해킹 의심 상황에서 **어느 신고 기관에 신고해야 하는지**를 정확하게 안내하기 어렵습니다.  
대신 제조사/서비스 제공 회사의 홈페이지, 등록 이메일, 공개된 담당자 연락처를 통해 먼저 보안취약점 정보와 대응 방법을 확인해 보세요.

### 참고 자료
- IoT 공통보안가이드, p.80


질문: 공유기 보안 설정을 안전하게 바꾸려면 어떻게 해야 하나요?



### 상황 판단
공유기는 집 안의 여러 기기를 인터넷에 연결해 주는 역할을 해서, 설정이 약하면 집 전체 보안에 영향을 줄 수 있습니다. 참고 자료에서는 초기 설정이 가장 안전하게 되어야 하고, 사용자가 초기값을 변경할 수 있도록 유도하는 것이 중요하다고 안내합니다.

### 지금 바로 할 수 있는 조치
1. 공유기 초기 설정값이 그대로인지 확인해 보세요.  
   - 처음 설치할 때의 기본 설정이 남아 있다면, 가능한 한 더 안전한 값으로 바꾸는 것이 좋습니다.

2. 공유기 관리자 비밀번호를 안전하게 바꾸세요.  
   - 참고 자료에서는 홈·가전 IoT 제품이 아이디/비밀번호 방식일 때 **안전한 비밀번호를 사용 및 설정**하도록 안내합니다.

3. 공유기에서 사용하지 않는 기능이 있는지 확인해 보세요.  
   - 참고 자료에는 **UPnP 등 취약한 서비스 제거**가 보안 점검 항목으로 나와 있습니다.

4. 펌웨어(기기 소프트웨어)가 최신인지 확인해 보세요.  
   - 참고 자료에는 **펌웨어 서명, 펌웨어 암호화, 펌웨어 다운그레이드 방지** 같은 보안 점검이 포함되어 있습니다.

### 보안 점검 체크리스트
- [ ] 초기 비밀번호를 바꿨는가
- [ ] 안전한 비밀번호를 사용하고 있는가
- [ ] 사용하지 않는 서비스(예: UPnP 등)를 껐는가
- [ ] 펌웨어 관련 보안이 적용되어 있는가
- [ ] 초기 설정이 “Secure by Default” 원칙처럼 안전하게 되어 있는가

### 추가 도움이 필요하면
현재 참고 자료만으로는 공유기 설정 화면에서 **어느 메뉴를 어떻게 눌러야 하는지**까지 정확히 안내하기 어렵습니다.  
만약 설정이 어렵다면 공유기 제조사 안내서나 고객센터 도움을 받는 것이 좋습니다.

### 참고 자료
- IoT 공통보안가이드, p.66
- IoT 공통보안가이드, p.54
- KISA 홈·가전 IoT 보안가이드, p.61

## 9. 추가 개선 아이디어 적용 확인

- 청킹 전략(500/100), 검색 개수(k=3)는 위 5~8번 셀에 이미 반영되어 실행됨
- 아래에서는 메타데이터 카테고리 필터링과 하이브리드 검색을 별도로 테스트

In [13]:
# 인덱스 존재 확인 후 없으면 생성
collection_info = client.get_collection(collection_name)
print("현재 인덱스:", collection_info.payload_schema)

if "metadata.category" not in collection_info.payload_schema:
    client.create_payload_index(
        collection_name=collection_name,
        field_name="metadata.category",
        field_schema="keyword"
    )
    print("✓ metadata.category 인덱스 생성 완료")
else:
    print("✓ 이미 인덱스가 존재합니다")

현재 인덱스: {}
✓ metadata.category 인덱스 생성 완료


## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [x] PDF 문서 선정 및 로딩 완료
- [x] Child Chunk 생성 완료
- [x] Qdrant Cloud에 데이터 저장 완료
- [x] Parent Document Retriever 구현 완료
- [x] 검색 테스트 완료 (Child vs Parent 비교)
- [x] RAG 시스템 구현 완료
- [x] 최소 3개 이상의 질문으로 테스트 완료
- [x] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합